# Train Best Passenger Counting Model (YOLOv8, 100 Epochs)

This notebook is configured to train **YOLOv8s** for **100 epochs** and report the **best-accuracy epoch** using validation metrics.


In [1]:
# Cell 1 - Environment and imports
import json
import time
from pathlib import Path

import pandas as pd
import torch
from ultralytics import YOLO

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.12.0.dev20260301+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
# Cell 2 - Configure fixed best model (YOLOv8) and 100-epoch training
PROJECT_ROOT = Path.cwd()

DEFAULT_SELECTION_JSON = (
    PROJECT_ROOT
    / "ai_models"
    / "passenger_counting"
    / "runs"
    / "detect"
    / "model_comparison"
    / "artifacts"
    / "best_model_selection.json"
)

DATA_YAML_CANDIDATES = [
    PROJECT_ROOT / "ai_models" / "passenger_counting" / "datasets" / "smart_bus_final" / "data.yaml",
    PROJECT_ROOT / "datasets" / "smart_bus_final" / "data.yaml",
]
DEFAULT_DATA_YAML = next((p for p in DATA_YAML_CANDIDATES if p.exists()), None)
if DEFAULT_DATA_YAML is None:
    checked = "\n".join(str(p) for p in DATA_YAML_CANDIDATES)
    raise FileNotFoundError(f"Dataset YAML not found. Checked:\n{checked}")

# Force model choice based on your confirmed result.
BEST_MODEL_NAME = "YOLOv8s"
BEST_BASE_WEIGHT = "yolov8s.pt"

# Optional: use dataset path from previous comparison artifact if available.
if DEFAULT_SELECTION_JSON.exists():
    with open(DEFAULT_SELECTION_JSON, "r", encoding="utf-8") as f:
        selection = json.load(f)
    DATA_YAML = Path(selection.get("dataset_yaml", str(DEFAULT_DATA_YAML)))
    print("Loaded dataset path from comparison artifact:", DEFAULT_SELECTION_JSON)
else:
    DATA_YAML = DEFAULT_DATA_YAML
    print("Comparison artifact not found. Using default dataset path.")

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Dataset YAML missing: {DATA_YAML}")

# Keep fixed at 100 as requested.
EPOCHS = 100
IMG_SIZE = 640
BATCH = 8
PATIENCE = 30
WORKERS = 4
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# Keep False to train from pretrained yolo8s.pt.
# Set True only if you intentionally want to continue from an existing checkpoint.
START_FROM_EXISTING_CHECKPOINT = False
EXISTING_CHECKPOINT_PATH = PROJECT_ROOT / "ai_models" / "passenger_counting" / "runs" / "detect" / "model_comparison" / "YOLOv8s" / "weights" / "best.pt"

RUN_NAME = f"{BEST_MODEL_NAME}_100_epochs"
FINAL_TRAIN_ROOT = PROJECT_ROOT / "ai_models" / "passenger_counting" / "runs" / "detect" / "best_model_100_epochs"
FINAL_TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

if START_FROM_EXISTING_CHECKPOINT and EXISTING_CHECKPOINT_PATH.exists():
    START_WEIGHTS = str(EXISTING_CHECKPOINT_PATH)
else:
    START_WEIGHTS = BEST_BASE_WEIGHT

print("Best model name:", BEST_MODEL_NAME)
print("Dataset:", DATA_YAML)
print("Start weights:", START_WEIGHTS)
print("Epochs:", EPOCHS)
print("Output root:", FINAL_TRAIN_ROOT)


Loaded dataset path from comparison artifact: C:\suttle project\smart-shuttle-ai-system\ai_models\passenger_counting\ai_models\passenger_counting\runs\detect\model_comparison\artifacts\best_model_selection.json
Best model name: YOLOv8s
Dataset: C:\suttle project\smart-shuttle-ai-system\ai_models\passenger_counting\datasets\smart_bus_final\data.yaml
Start weights: yolov8s.pt
Epochs: 100
Output root: C:\suttle project\smart-shuttle-ai-system\ai_models\passenger_counting\ai_models\passenger_counting\runs\detect\best_model_100_epochs


In [3]:
# Cell 3 - Train YOLOv8 for 100 epochs and capture best-accuracy epoch
start_time = time.time()

model = YOLO(START_WEIGHTS)
train_result = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    device=DEVICE,
    workers=WORKERS,
    cache=True,
    optimizer="AdamW",
    lr0=0.001,
    project=str(FINAL_TRAIN_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    pretrained=True,
    verbose=True,
)

elapsed_min = round((time.time() - start_time) / 60, 2)
run_dir = Path(getattr(train_result, "save_dir", FINAL_TRAIN_ROOT / RUN_NAME))
best_pt = run_dir / "weights" / "best.pt"
results_csv = run_dir / "results.csv"

# Evaluate best checkpoint directly for reliable final metrics.
if best_pt.exists():
    best_model = YOLO(str(best_pt))
else:
    best_model = model

best_val_result = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    verbose=False,
)

best_checkpoint_metrics = {
    "precision": float(best_val_result.box.mp) if hasattr(best_val_result, "box") else 0.0,
    "recall": float(best_val_result.box.mr) if hasattr(best_val_result, "box") else 0.0,
    "mAP50": float(best_val_result.box.map50) if hasattr(best_val_result, "box") else 0.0,
    "mAP50-95": float(best_val_result.box.map) if hasattr(best_val_result, "box") else 0.0,
}

best_epoch_summary = {
    "best_epoch": None,
    "best_epoch_metrics": {},
}

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]

    epoch_col = "epoch" if "epoch" in df.columns else None
    map_col_candidates = ["metrics/mAP50-95(B)", "metrics/mAP50-95"]
    precision_col_candidates = ["metrics/precision(B)", "metrics/precision"]
    recall_col_candidates = ["metrics/recall(B)", "metrics/recall"]
    map50_col_candidates = ["metrics/mAP50(B)", "metrics/mAP50"]

    map_col = next((c for c in map_col_candidates if c in df.columns), None)
    p_col = next((c for c in precision_col_candidates if c in df.columns), None)
    r_col = next((c for c in recall_col_candidates if c in df.columns), None)
    m50_col = next((c for c in map50_col_candidates if c in df.columns), None)

    if map_col is not None and len(df) > 0:
        best_idx = df[map_col].astype(float).idxmax()
        row = df.loc[best_idx]

        best_epoch_value = int(row[epoch_col]) if epoch_col else int(best_idx + 1)
        best_epoch_summary = {
            "best_epoch": best_epoch_value,
            "best_epoch_metrics": {
                "precision": round(float(row[p_col]), 6) if p_col else None,
                "recall": round(float(row[r_col]), 6) if r_col else None,
                "mAP50": round(float(row[m50_col]), 6) if m50_col else None,
                "mAP50-95": round(float(row[map_col]), 6),
            },
        }

training_summary = {
    "best_model_name": BEST_MODEL_NAME,
    "start_weights": START_WEIGHTS,
    "epochs": EPOCHS,
    "dataset_yaml": str(DATA_YAML),
    "run_dir": str(run_dir),
    "best_checkpoint": str(best_pt) if best_pt.exists() else "Not found",
    "results_csv": str(results_csv) if results_csv.exists() else "Not found",
    "elapsed_minutes": elapsed_min,
    "best_checkpoint_metrics": best_checkpoint_metrics,
    "best_epoch": best_epoch_summary["best_epoch"],
    "best_epoch_metrics_from_results_csv": best_epoch_summary["best_epoch_metrics"],
}

print("Training complete in", elapsed_min, "minutes")
print("Run dir:", run_dir)
print("Best checkpoint:", training_summary["best_checkpoint"])
print("Best checkpoint metrics:", best_checkpoint_metrics)
print("Best accuracy epoch:", training_summary["best_epoch"])
print("Best epoch metrics (from results.csv):", training_summary["best_epoch_metrics_from_results_csv"])


Ultralytics 8.4.41  Python-3.11.9 torch-2.12.0.dev20260301+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\suttle project\smart-shuttle-ai-system\ai_models\passenger_counting\datasets\smart_bus_final\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, 

In [4]:
# Cell 4 - Save training summary
summary_path = FINAL_TRAIN_ROOT / f"{RUN_NAME}_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(training_summary, f, indent=2)

print("Saved:", summary_path)


Saved: C:\suttle project\smart-shuttle-ai-system\ai_models\passenger_counting\ai_models\passenger_counting\runs\detect\best_model_100_epochs\YOLOv8s_100_epochs_summary.json
